# Teste de remoção da feature `material_coil_support` — Oven HMI e CT Knobs

Este notebook responde a uma pergunta única e bem delimitada: **remover a feature `material_coil_support` das áreas Oven HMI e CT Knobs melhora, piora ou não muda o erro do modelo?**

## A hipótese

A feature `material_coil_support` descreve o material do suporte da bobina de indução (alumínio ou plástico). A hipótese é que esse material **não tenha efeito físico relevante** nas áreas Oven HMI (interface do forno) e CT Knobs (botões do cooktop), por serem regiões distantes da bobina.

Se a hipótese estiver certa, essas duas colunas são apenas ruído: ocupam espaço no vetor de entrada sem carregar sinal. Removê-las melhora a proporção entre a quantidade de dados e a quantidade de colunas, o que tende a ajudar um modelo treinado sobre uma base pequena.

## Como o teste é feito

Para cada uma das duas áreas, o **mesmo modelo** é rodado duas vezes:

1. **Baseline** — com todas as features.
2. **Sem material** — removendo as **duas** colunas `material_coil_support_aluminium` e `material_coil_support_plastic` **juntas**.

> **Por que as duas juntas?** As colunas são uma codificação *one-hot*: elas representam uma única variável categórica repartida em duas colunas indicadoras. Remover só uma delas não apagaria a informação — a coluna restante continuaria dizendo exatamente a mesma coisa, só que invertida. Pior: quebraria a codificação, deixando uma variável pela metade. O teste só faz sentido removendo o par inteiro.

A comparação é feita sob **exatamente o mesmo protocolo** da entrega da Sprint 3: mesmo modelo, mesma partição, mesmas dobras. Só assim a diferença observada pode ser atribuída à feature, e não a outra mudança qualquer.

## O que não é feito aqui

O **conjunto de teste reservado não é tocado**. Todas as métricas deste notebook vêm da validação cruzada dentro do conjunto de treino.

## 1. Configuração

O notebook depende de três arquivos, todos da pasta `notebooks/colabs_S3/`:

| Arquivo | Para que serve |
|---|---|
| `dataset_oven_hmi_13cat.csv` | Dados da área Oven HMI |
| `dataset_ct_knobs_13cat.csv` | Dados da área CT Knobs |
| `particao_treino_teste.csv` | Quais ensaios são de treino/teste e a que dobra cada um pertence |

São usados **apenas** os arquivos com sufixo `_13cat` (as versões `_perfil` e `_snull` são outras variantes de tratamento e não entram neste teste).

No Google Colab, suba os três arquivos pela barra lateral (ícone de pasta) e rode a célula abaixo. No repositório local, a função `resolve()` encontra os caminhos sozinha.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4/testes_extras/testes_S4/testesCoilEOven"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import os

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

SUFIXO = "_13cat"
PASTA_DADOS = "colabs_S3"

AREAS = ["oven_hmi", "ct_knobs"]

# As duas colunas que formam a codificacao one-hot da feature sob teste.
# Elas so podem ser removidas juntas: sao uma variavel categorica unica.
COLUNAS_MATERIAL = ["material_coil_support_aluminium", "material_coil_support_plastic"]

BASES = [
    ".", "..", os.path.join("..", ".."), os.path.join("..", "..", ".."), os.path.join("..", "..", "..", ".."), "/content",
    "notebooks", os.path.join("..", "notebooks"),
    os.path.join("notebooks", PASTA_DADOS),
]


def resolve(arquivo, pasta=PASTA_DADOS):
    """Encontra o arquivo no Colab ou no repositorio local."""
    tentativas = []
    # A pasta declarada tem prioridade sobre o arquivo solto: sem isso, um
    # homonimo numa pasta vizinha pode ser lido no lugar do arquivo pedido.
    for partes in ((pasta, arquivo), (arquivo,)):
        for base in BASES:
            caminho = os.path.join(base, *partes)
            tentativas.append(caminho)
            if os.path.exists(caminho):
                return caminho
    raise FileNotFoundError(
        "Arquivo nao encontrado: " + arquivo
        + ". Diretorio atual: " + os.path.abspath(".")
        + ". Caminhos tentados: " + "; ".join(tentativas)
    )


print("Diretorio atual:", os.path.abspath("."))
print("Versao dos datasets:", SUFIXO)
print("Areas sob teste:", ", ".join(AREAS))

## 2. Leitura dos dados e da partição

Os datasets já vêm com o *One-Hot Encoding* aplicado pelos notebooks de tratamento. Cada linha é um ensaio; as colunas se dividem em dois grupos:

- **Features** — identificadas pelos prefixos `blower_direction_`, `housing_`, `board_position_`, `run_case_`, `losses_`, `deflector_hole_` e `material_coil_support_`.
- **Colunas-alvo** — uma por componente, contendo a temperatura medida em graus Celsius. É aqui que estão os valores ausentes, quando um sensor não registrou naquele ensaio.

A partição vem de `particao_treino_teste.csv`, com três colunas relevantes: `run_idx` (o ensaio), `conjunto` (treino ou teste) e `dobra` (o grupo de validação cruzada, preenchido só para os ensaios de treino).

In [ ]:
PREFIXOS_FEATURES = [
    "blower_direction", "housing", "board_position", "run_case",
    "losses", "deflector_hole", "material_coil_support",
]
COLUNAS_AUXILIARES = ["run_idx"]

datasets = {}
for nome in AREAS:
    datasets[nome] = pd.read_csv(resolve(f"dataset_{nome}{SUFIXO}.csv"))

particao = pd.read_csv(resolve("particao_treino_teste.csv"))

# Mapa run_idx -> dobra. Ensaios de teste ficam com dobra nula e, por isso,
# sao naturalmente excluidos de toda a validacao mais adiante.
DOBRA = {int(r.run_idx): (None if pd.isna(r.dobra) else int(r.dobra))
         for r in particao.itertuples()}
DOBRAS = sorted({d for d in DOBRA.values() if d is not None})

IDX_TREINO = [int(r.run_idx) for r in particao.itertuples() if r.conjunto == "treino"]
IDX_TESTE = [int(r.run_idx) for r in particao.itertuples() if r.conjunto == "teste"]

print(f"Ensaios de treino: {len(IDX_TREINO)}  |  Ensaios de teste: {len(IDX_TESTE)}")
print(f"Dobras de validacao cruzada: {len(DOBRAS)} -> {DOBRAS}")
print()

for nome, df in datasets.items():
    tem_run_idx = "run_idx" in df.columns
    print(f"{nome:<10} {df.shape[0]:>3} linhas x {df.shape[1]:>3} colunas"
          f"   run_idx no arquivo: {'sim' if tem_run_idx else 'nao'}")

### Conferência: onde estão os valores ausentes e como o `run_idx` é resolvido

Duas verificações importam antes de modelar.

**Primeira: os nulos estão só nas colunas-alvo.** Se houvesse nulo em alguma feature, seria preciso imputar — e a imputação teria de ser calculada só sobre o treino de cada dobra, nunca sobre a base toda, sob pena de vazar informação da validação para o treino. A célula abaixo confirma que não é o caso: os ausentes estão todos nas colunas de temperatura, e a essas o modelo simplesmente **não é exposto** — as linhas sem medição são descartadas, nunca preenchidas. Assim, a avaliação recai sempre sobre valor medido de verdade, jamais sobre valor imputado.

**Segunda: como ligar cada linha ao seu `run_idx`.** Os datasets `_13cat` não trazem a coluna `run_idx`. A convenção usada na entrega — e repetida aqui — é que, na ausência dela, o índice posicional da linha é o próprio `run_idx`. A célula verifica que o número de linhas bate com o número de ensaios da partição, que é o que torna essa convenção válida.

In [ ]:
def separar(df, prefixos=PREFIXOS_FEATURES):
    """Devolve (X, cols_alvo).

    Uma coluna e feature quando comeca por um dos prefixos declarados; e alvo
    caso contrario. As colunas auxiliares (run_idx) nao entram em nenhum dos
    dois grupos: servem apenas para indexar a particao.
    """
    cols_entrada = [c for c in df.columns if any(c.startswith(p + "_") for p in prefixos)]
    cols_alvo = [c for c in df.columns if c not in cols_entrada + COLUNAS_AUXILIARES]
    return df[cols_entrada].copy(), cols_alvo


N_ENSAIOS = len(particao)
ok_geral = True

for nome, df in datasets.items():
    X, cols_alvo = separar(df)

    nulos_em_features = int(X.isna().sum().sum())
    nulos_em_alvos = int(df[cols_alvo].isna().sum().sum())
    linhas_ok = len(df) == N_ENSAIOS
    material_ok = all(c in X.columns for c in COLUNAS_MATERIAL)

    print(f"--- {nome} ---")
    print(f"  features: {X.shape[1]}   |   componentes (alvos): {len(cols_alvo)}")
    print(f"  nulos nas features: {nulos_em_features}   |   nulos nos alvos: {nulos_em_alvos}")
    print(f"  linhas == ensaios da particao ({N_ENSAIOS}): {'sim' if linhas_ok else 'NAO'}")
    print(f"  as duas colunas de material estao presentes: {'sim' if material_ok else 'NAO'}")
    print(f"  componentes: {', '.join(cols_alvo)}")
    print()

    if nulos_em_features or not linhas_ok or not material_ok:
        ok_geral = False

print("Verificacao:", "PASSOU" if ok_geral else "FALHOU")
print()
print("Nenhum nulo nas features -> nao ha imputacao a fazer. Os nulos dos alvos sao")
print("tratados por descarte da linha, de modo que toda metrica recai sobre valor medido.")

## 3. Métricas de avaliação

As três métricas do teste, calculadas sobre as predições acumuladas de todas as dobras:

- **MAE** (erro absoluto médio) — de quanto o modelo erra, em média, em graus Celsius. É a métrica principal deste teste: está na unidade do problema e é fácil de interpretar.
- **RMSE** (raiz do erro quadrático médio) — na mesma unidade, mas penalizando mais os erros grandes. Um RMSE muito acima do MAE indica que os erros estão concentrados em poucos casos ruins.
- **R²** — fração da variação do alvo que o modelo explica. Zero significa que o modelo não é melhor que chutar sempre a média; negativo, que é pior que isso.

A definição é a mesma do notebook `modelagem.ipynb`, com dois cuidados herdados de lá: a conversão para `numpy` antes de subtrair (duas `Series` do pandas se alinham por índice, não por posição, e isso produziria diferenças entre pares errados, em silêncio) e o cálculo explícito da raiz no RMSE (o parâmetro que a produzia foi removido do scikit-learn na versão 1.9).

In [ ]:
def avaliar(y_real, y_previsto):
    """MAE, RMSE e R2 de um conjunto de predicoes, com o alvo em graus Celsius."""
    y_real = np.asarray(y_real, dtype=float).ravel()
    y_previsto = np.asarray(y_previsto, dtype=float).ravel()

    if y_real.shape != y_previsto.shape:
        raise ValueError(f"dimensoes diferentes: {y_real.shape} e {y_previsto.shape}")
    if y_real.size == 0:
        raise ValueError("nenhuma observacao a avaliar")
    if np.isnan(y_real).any():
        raise ValueError("ha valores ausentes em y_real; a mascara deve ser aplicada antes")
    if np.isnan(y_previsto).any():
        raise ValueError("ha valores ausentes em y_previsto")

    erro = y_previsto - y_real
    mae = float(np.mean(np.abs(erro)))
    rmse = float(np.sqrt(np.mean(erro ** 2)))

    if np.unique(y_real).size < 2:
        r2 = float("nan")
    else:
        ss_res = float(np.sum(erro ** 2))
        ss_tot = float(np.sum((y_real - np.mean(y_real)) ** 2))
        r2 = 1.0 - ss_res / ss_tot

    return {"n": int(y_real.size), "MAE": mae, "RMSE": rmse, "R2": r2}


print("Metricas definidas: MAE, RMSE, R2.")

## 4. O modelo: um Random Forest por área, com patamar e escala

Esta é a formulação adotada na entrega da Sprint 3 (Seção 19 do `modelagem.ipynb`), reproduzida aqui sem alteração. Vale explicar o que ela faz, porque não é um Random Forest comum.

### Um modelo por área, e não por componente

Em vez de treinar um modelo separado para cada componente, treina-se **um único modelo por área**. Para isso, os dados são reorganizados em **formato longo**: cada linha passa a ser um par *(ensaio, componente)*. A área Oven HMI, por exemplo, com 4 componentes e 56 ensaios, vira uma tabela de até 224 linhas em vez de 4 tabelas de 56.

### Os dois atributos do componente: patamar e escala

Se o modelo recebesse só o vetor de configuração, ele não teria como saber de qual componente é cada linha — e componentes diferentes operam em temperaturas muito diferentes. Por isso, ao vetor de features somam-se **dois atributos que descrevem o componente**:

- **patamar** — a temperatura média daquele componente;
- **escala** — o desvio-padrão daquele componente, isto é, o quanto ele varia em resposta à configuração.

O patamar diz ao modelo *em que faixa* aquele componente vive; a escala diz *o quanto ele se mexe*. Com isso, o modelo não gasta capacidade aprendendo o patamar de cada componente a partir de colunas indicadoras e pode usá-la para aprender o efeito da configuração — que é o que de fato interessa.

> **O ponto crítico:** patamar e escala são calculados **exclusivamente sobre as linhas de treino de cada dobra**. Se fossem calculados sobre a base inteira, carregariam informação dos ensaios de validação para dentro do treino, e o erro medido sairia otimista — o modelo estaria, em parte, consultando a resposta.

O Random Forest usa 300 árvores, com os demais parâmetros no padrão do scikit-learn — inclusive `max_features`, que a entrega deliberadamente não altera.

In [ ]:
# Parametros identicos aos da entrega (Secao 19 do modelagem.ipynb).
AREA_PARAMS = dict(n_estimators=300, random_state=42, n_jobs=-1)

MIN_TREINO = 8      # linhas minimas de treino para a dobra valer
MIN_PREDICOES = 5   # medicoes minimas para o componente entrar no teste


def formato_longo(df, colunas_excluidas=()):
    """Uma linha por par de ensaio e componente medido.

    colunas_excluidas permite retirar features do vetor de entrada -- e por
    onde a rodada sem material_coil_support passa a sua remocao.
    """
    X, cols_alvo = separar(df)
    X = X.drop(columns=list(colunas_excluidas))

    # Sem a coluna run_idx no arquivo, o indice posicional e o proprio run_idx.
    run = (df["run_idx"].to_numpy() if "run_idx" in df.columns
           else np.arange(len(df)))

    blocos = []
    for col in cols_alvo:
        y = df[col]
        if y.notna().sum() < MIN_PREDICOES:
            continue
        medido = y.notna()
        sub = X[medido].copy()
        sub["_run"] = run[medido.to_numpy()]
        sub["_componente"] = col
        sub["_alvo"] = y[medido].to_numpy()
        blocos.append(sub)

    longo = pd.concat(blocos, ignore_index=True)
    longo["_dobra"] = longo["_run"].map(DOBRA)
    # Linhas sem dobra sao os ensaios do conjunto de teste: saem aqui e nao
    # voltam. O conjunto reservado nao participa de nada neste notebook.
    return longo[longo["_dobra"].notna()].reset_index(drop=True)


def atributos_do_componente(longo, treino):
    """Patamar e escala de cada componente, medidos SO no treino da dobra."""
    g = longo[treino].groupby("_componente")["_alvo"]
    media = g.mean()
    desvio = g.std().fillna(1.0).replace(0, 1.0)
    geral = float(longo.loc[treino, "_alvo"].mean())
    return media, desvio, geral


def matriz(longo, mascara, entradas, media, desvio, geral):
    """Vetor de configuracao acrescido do patamar e da escala do componente."""
    comp = longo.loc[mascara, "_componente"]
    return np.hstack([
        longo.loc[mascara, entradas].to_numpy(),
        comp.map(media).fillna(geral).to_numpy().reshape(-1, 1),
        comp.map(desvio).fillna(1.0).to_numpy().reshape(-1, 1),
    ])


print("Formulacao por area definida (Random Forest, 300 arvores, patamar + escala).")

## 5. A validação: *leave-one-group-out* sobre as configurações de treino

A validação segue exatamente o protocolo da entrega. Cada **dobra** da partição agrupa os ensaios que compartilham a mesma configuração física. A validação percorre uma dobra por vez: essa dobra fica de fora, o modelo é treinado no restante e prevê os valores dela.

Agrupar por configuração, em vez de sortear linhas soltas, é o que torna a medida honesta. Ensaios da mesma configuração são quase réplicas um do outro — se um deles ficasse no treino e outro na validação, o modelo estaria praticamente vendo a resposta, e o erro medido seria bom demais para ser verdade.

As predições de todas as dobras são acumuladas e as métricas calculadas **uma única vez** sobre o conjunto todo, em vez de calculadas por dobra e depois mediadas. Isso evita a instabilidade do R² quando uma dobra tem poucos pontos — situação comum aqui, dado o tamanho da base.

A função abaixo executa uma rodada completa para uma área e devolve as predições de validação, linha a linha.

In [ ]:
def rodar_area(nome, df, colunas_excluidas=()):
    """Validacao leave-one-group-out completa de uma area.

    Devolve um DataFrame com uma linha por predicao de validacao, contendo o
    componente, o ensaio, o valor real e o previsto.
    """
    longo = formato_longo(df, colunas_excluidas)
    entradas = [c for c in longo.columns if not c.startswith("_")]

    predicoes = []
    for d in DOBRAS:
        treino = (longo["_dobra"] != d).to_numpy()
        val = (longo["_dobra"] == d).to_numpy()
        if treino.sum() < MIN_TREINO or not val.any():
            continue

        # Patamar e escala: so o treino desta dobra entra no calculo.
        media, desvio, geral = atributos_do_componente(longo, treino)
        X_tr = matriz(longo, treino, entradas, media, desvio, geral)
        X_val = matriz(longo, val, entradas, media, desvio, geral)

        # A padronizacao tambem e ajustada so no treino da dobra. Ela e mantida
        # por uniformidade de protocolo com a entrega, ainda que a floresta
        # aleatoria seja invariante a escala.
        escala = StandardScaler()
        modelo = RandomForestRegressor(**AREA_PARAMS)
        modelo.fit(escala.fit_transform(X_tr), longo.loc[treino, "_alvo"])
        previsto = modelo.predict(escala.transform(X_val))

        bloco = longo.loc[val, ["_componente", "_run", "_alvo"]].copy()
        bloco.columns = ["componente", "ensaio", "real"]
        bloco["previsto"] = previsto
        bloco["dobra"] = d
        predicoes.append(bloco)

    saida = pd.concat(predicoes, ignore_index=True)
    saida["area"] = nome
    saida["erro"] = saida["previsto"] - saida["real"]
    return saida


def metricas_por_area(predicoes):
    """MAE, RMSE e R2 medios sobre os componentes da area.

    Cada componente e avaliado separadamente e as tres metricas sao entao
    promediadas entre eles, de modo que todo componente pese igual --
    inclusive os que tem menos medicoes.
    """
    por_componente = []
    for comp, g in predicoes.groupby("componente"):
        por_componente.append({"componente": comp,
                               **avaliar(g["real"].to_numpy(), g["previsto"].to_numpy())})
    por_componente = pd.DataFrame(por_componente)
    return {
        "componentes": len(por_componente),
        "n": int(por_componente["n"].sum()),
        "MAE": float(por_componente["MAE"].mean()),
        "RMSE": float(por_componente["RMSE"].mean()),
        "R2": float(por_componente["R2"].mean()),
    }, por_componente


print("Protocolo de validacao definido.")

## 6. As quatro rodadas

Agora o teste propriamente dito: duas áreas × duas condições (com e sem a feature), quatro rodadas no total.

Note que a **única** diferença entre a rodada baseline e a rodada "sem material" é o argumento `colunas_excluidas`. Todo o resto — partição, dobras, semente aleatória, número de árvores, cálculo do patamar e da escala — é idêntico. É essa identidade que permite atribuir a diferença observada à feature, e não a alguma outra variação.

In [ ]:
CONDICOES = [
    ("baseline", ()),
    ("sem_material", tuple(COLUNAS_MATERIAL)),
]

resultados = []
predicoes_todas = {}
detalhe_por_componente = {}

for nome in AREAS:
    df = datasets[nome]
    for rotulo, excluidas in CONDICOES:
        predicoes = rodar_area(nome, df, excluidas)
        resumo, por_comp = metricas_por_area(predicoes)

        predicoes_todas[(nome, rotulo)] = predicoes
        detalhe_por_componente[(nome, rotulo)] = por_comp

        n_features = len([c for c in formato_longo(df, excluidas).columns
                          if not c.startswith("_")])
        resultados.append({"area": nome, "condicao": rotulo,
                           "features": n_features, **resumo})

        print(f"=== {nome}  |  {rotulo} ===")
        if excluidas:
            print(f"    colunas removidas: {', '.join(excluidas)}")
        else:
            print("    colunas removidas: nenhuma")
        print(f"    features no vetor de configuracao: {n_features} "
              f"(+ 2 atributos do componente: patamar e escala)")
        print(f"    componentes: {resumo['componentes']}   |   "
              f"predicoes de validacao: {resumo['n']}")
        print(f"    MAE  {resumo['MAE']:.3f} C")
        print(f"    RMSE {resumo['RMSE']:.3f} C")
        print(f"    R2   {resumo['R2']:+.3f}")
        print()

df_resultados = pd.DataFrame(resultados)

## 7. Tabela comparativa

A tabela abaixo confronta as duas condições, área por área. A coluna que responde à pergunta é o **Δ MAE**, a diferença entre o MAE sem a feature e o MAE com ela:

- **Δ MAE negativo** → o erro caiu ao remover a feature. Remover **ajudou**.
- **Δ MAE positivo** → o erro subiu ao remover a feature. A feature **importava**.
- **Δ MAE perto de zero** → a feature é indiferente. Nesse caso, o critério de desempate é a parcimônia: entre dois modelos equivalentes, prefere-se o de menor dimensionalidade.

In [ ]:
tabela = df_resultados.pivot(index="area", columns="condicao",
                             values=["MAE", "RMSE", "R2", "features"])
tabela = tabela.reindex(AREAS)

comparacao = pd.DataFrame({
    "features_baseline": tabela[("features", "baseline")].astype(int),
    "features_sem_material": tabela[("features", "sem_material")].astype(int),
    "MAE_baseline": tabela[("MAE", "baseline")],
    "MAE_sem_material": tabela[("MAE", "sem_material")],
    "delta_MAE": tabela[("MAE", "sem_material")] - tabela[("MAE", "baseline")],
    "RMSE_baseline": tabela[("RMSE", "baseline")],
    "RMSE_sem_material": tabela[("RMSE", "sem_material")],
    "delta_RMSE": tabela[("RMSE", "sem_material")] - tabela[("RMSE", "baseline")],
    "R2_baseline": tabela[("R2", "baseline")],
    "R2_sem_material": tabela[("R2", "sem_material")],
    "delta_R2": tabela[("R2", "sem_material")] - tabela[("R2", "baseline")],
})

comparacao["veredito"] = np.where(
    comparacao["delta_MAE"] < 0, "remover ajudou",
    np.where(comparacao["delta_MAE"] > 0, "a feature importava", "indiferente"))

print("Comparacao baseline x sem material_coil_support")
print("(delta = sem material - baseline; negativo significa que remover reduziu o erro)")
print()
print(comparacao.round(3).to_string())

## 8. Leitura do resultado

A célula abaixo traduz os números em palavras, área por área, e registra a ressalva que qualquer leitura desses números precisa carregar.

In [ ]:
print("=" * 72)
print("CONCLUSAO DO TESTE")
print("=" * 72)
print()

for area in AREAS:
    linha = comparacao.loc[area]
    d = linha["delta_MAE"]
    print(f"{area}")
    print(f"  MAE  {linha['MAE_baseline']:.3f} C (baseline)  ->  "
          f"{linha['MAE_sem_material']:.3f} C (sem material)   delta {d:+.3f} C")
    if d < 0:
        print(f"  Remover a feature REDUZIU o erro em {abs(d):.3f} C.")
        print("  Compativel com a hipotese: a area e distante da bobina e as duas colunas")
        print("  ocupavam espaco no vetor de entrada sem carregar sinal.")
    elif d > 0:
        print(f"  Remover a feature AUMENTOU o erro em {d:.3f} C.")
        print("  Contraria a hipotese: a feature carregava sinal util para esta area.")
    else:
        print("  O erro nao mudou. A feature e indiferente para esta area.")
    print(f"  Delta RMSE {linha['delta_RMSE']:+.3f} C   |   "
          f"delta R2 {linha['delta_R2']:+.3f}")
    print()

print("-" * 72)
print("Ressalva sobre o alcance destes numeros")
print("-" * 72)
print("As metricas vem da validacao cruzada dentro do conjunto de treino. O conjunto")
print("de teste reservado nao foi consultado, de modo que este resultado nao foi")
print("confirmado em dado independente.")
print()
print("A base e pequena e a floresta aleatoria e estocastica: diferencas de poucos")
print("centesimos de grau estao dentro da variacao do proprio procedimento e nao")
print("sustentam decisao. Um delta dessa ordem deve ser lido como empate, e nao como")
print("vitoria de uma das condicoes.")

### Detalhe por componente

Por fim, o desdobramento componente a componente. Ele mostra se o efeito observado na média da área é homogêneo ou se vem de um componente isolado — informação que a média sozinha esconde.

In [ ]:
for area in AREAS:
    base = detalhe_por_componente[(area, "baseline")].set_index("componente")
    sem = detalhe_por_componente[(area, "sem_material")].set_index("componente")

    det = pd.DataFrame({
        "n": base["n"],
        "MAE_baseline": base["MAE"],
        "MAE_sem_material": sem["MAE"],
        "delta_MAE": sem["MAE"] - base["MAE"],
        "R2_baseline": base["R2"],
        "R2_sem_material": sem["R2"],
    })
    melhoram = int((det["delta_MAE"] < 0).sum())

    print(f"--- {area} ---")
    print(det.round(3).to_string())
    print(f"  remover a feature melhora {melhoram} de {len(det)} componentes")
    print()

## 9. O Δ observado é maior que o ruído do próprio procedimento?

Esta seção existe por um motivo simples: os Δ das seções anteriores são pequenos, e um número pequeno com sinal definido ainda pode ser apenas acaso.

O Random Forest é **estocástico** — ele sorteia amostras e colunas ao construir cada árvore. Trocar a semente aleatória muda um pouco o resultado, mesmo com dados e protocolo idênticos. Se o Δ que medimos for da mesma ordem dessa variação, ele não diz nada sobre a feature: diz apenas que o modelo é aleatório.

O teste abaixo repete as quatro rodadas com **várias sementes** e compara duas quantidades:

- o **Δ MAE médio** entre as sementes (o efeito da feature, se houver);
- o **desvio-padrão do Δ entre as sementes** (o ruído do procedimento).

A leitura é direta: se o Δ médio não for maior que esse desvio, as duas condições são **indistinguíveis**, e a conclusão correta é empate — não vitória de nenhum dos lados.

Cada semente refaz a validação inteira nas duas áreas, então a célula demora alguns minutos. Para pular, mude `SEMENTES` para uma lista de um elemento só.

In [ ]:
SEMENTES = [42, 7, 123, 2024, 31337]

linhas_semente = []
for semente in SEMENTES:
    AREA_PARAMS["random_state"] = semente
    for nome in AREAS:
        df = datasets[nome]
        mae = {}
        for rotulo, excluidas in CONDICOES:
            resumo, _ = metricas_por_area(rodar_area(nome, df, excluidas))
            mae[rotulo] = resumo["MAE"]
        linhas_semente.append({
            "semente": semente, "area": nome,
            "MAE_baseline": mae["baseline"],
            "MAE_sem_material": mae["sem_material"],
            "delta_MAE": mae["sem_material"] - mae["baseline"],
        })

AREA_PARAMS["random_state"] = 42   # restaura a semente da entrega

por_semente = pd.DataFrame(linhas_semente)

print(f"Delta do MAE por semente ({len(SEMENTES)} sementes):")
print()
print(por_semente.pivot(index="semente", columns="area", values="delta_MAE")
      .reindex(columns=AREAS).round(3).to_string())
print()

print("=" * 72)
for area in AREAS:
    d = por_semente[por_semente["area"] == area]["delta_MAE"]
    media, desvio = float(d.mean()), float(d.std(ddof=1))
    positivos = int((d > 0).sum())

    print(f"{area}")
    print(f"  delta MAE medio entre sementes : {media:+.3f} C")
    print(f"  desvio entre sementes (ruido)  : {desvio:.3f} C")
    print(f"  o delta e positivo em {positivos} de {len(d)} sementes")
    if abs(media) > desvio:
        direcao = "AUMENTOU" if media > 0 else "REDUZIU"
        print(f"  -> O efeito supera o ruido do procedimento: remover a feature {direcao}")
        print("     o erro de forma consistente entre as sementes.")
    else:
        print("  -> O efeito NAO supera o ruido do procedimento. As duas condicoes sao")
        print("     indistinguiveis nesta area: o resultado e empate, e o sinal do delta")
        print("     de uma rodada isolada nao sustenta conclusao sobre a feature.")
    print()

print("-" * 72)
print("Como decidir em caso de empate")
print("-" * 72)
print("Empate nao significa ausencia de resposta. Se as duas condicoes erram igual, o")
print("criterio de desempate e a parcimonia: entre dois modelos equivalentes, prefere-se")
print("o de menor dimensionalidade, que e o que melhora a proporcao entre dados e")
print("colunas. E esse o criterio ja registrado na Secao 4.3.1 da documentacao, aplicado")
print("quando a versao de 13 categorias empatou com a de perfil de perdas.")